# MMRL / BayesMMRL 动态 R 权重融合验证（repo-forward + 真实日志配置恢复版）

本文件修复 `mmrl_dynamic_r_fusion_repo_forward_corrected.ipynb` 的关键问题：

> **不再只从 `case_root` 路径推断配置。**  
> 每个 case 必须从真实训练日志 `run.log` / `log.txt` / `log.txt-*` 的 `** Arguments **` 区块恢复 `dataset_config_file`、`method_config_file`、`protocol_config_file`、`runtime_config_file`、`exp_config`、`exec_mode` 和完整 `opts`。

这很重要，因为 MMRL / BayesMMRL 的 checkpoint 是 lightweight checkpoint，很多行为由当前重建的 cfg 决定。只靠路径恢复 `shots/backbone/subsample` 会漏掉真实训练时的 `BAYES_MMRL.*`、`MMRL.*`、`EVAL_AGGREGATION`、`N_MC_TEST`、`ALPHA` 等覆盖项，导致 z_c / z_r / fusion 的准确率异常。

核心提取路径仍然严格使用 repo 标准评估接口：

```python
outputs = trainer.method.forward_eval(batch, eval_ctx)
z_c = outputs.logits
z_r = outputs.aux_logits["rep"]
z_aux_fusion = outputs.aux_logits["fusion"]
z_repo_eval = trainer.method.select_eval_logits(outputs, eval_ctx)
```

输出文件：

```text
output_refactor/analysis/dynamic_r_fusion_repo_forward_log_restored/
  dynamic_r_fusion_repo_forward_log_restored_report.xlsx
```


In [1]:
from pathlib import Path

# ====== 改这里 ======
REPO_ROOT = Path("/root/autodl-tmp/MMRL").expanduser().resolve()
DATA_ROOT = REPO_ROOT / "DATASETS"

# 每个 case_root 必须是一次真实实验的 seed 目录。
# notebook 会从该目录下的 run.log / log.txt / log.txt-* 恢复真实 Arguments/opts。
CASES = [
    {
        "name": "MMRL_16shot_seed1",
        "case_root": REPO_ROOT / "output_refactor/MMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/default/seed1",
        # 可选：显式指定真实日志文件；不填则自动找 run.log / log.txt / log.txt-*。
        # "log_file": REPO_ROOT / ".../seed1/run.log",
        # 可选：指定加载 epoch；默认加载 model-best 或最后一个 model.pth。
        # "load_epoch": None,
    },
    {
        "name": "BayesMMRL_16shot_seed1",
        "case_root": REPO_ROOT / "output_refactor/BayesMMRL/FS/fewshot_train/caltech101/shots_16/ViT-B-16/default/seed1",
        # "log_file": REPO_ROOT / ".../seed1/run.log",
    },
]

TEST_SPLIT = "test"
BETA_SELECT_SPLIT = "val"

# 调试时可设成小整数；正式实验设 None。
MAX_BATCHES = None

# β 网格。动态融合部分仅用于分析；分支提取和准确率不依赖 β。
BETA_GRID_VALUES = [0.0, 0.5, 1.0, 2.0, 4.0]

# 固定 BayesMMRL MC eval 随机种子，降低重复运行波动。
EVAL_SEED = 2026

# 本版建议每次重新收集，避免读到旧 notebook 生成的错误缓存。
SAVE_NPZ_CACHE = True
USE_NPZ_CACHE_IF_EXISTS = False

# 关键开关：必须从真实日志恢复配置。
REQUIRE_REAL_LOG_CONFIG = True

OUT_DIR = REPO_ROOT / "output_refactor/analysis/dynamic_r_fusion_repo_forward_log_restored"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert REPO_ROOT.exists(), f"REPO_ROOT 不存在: {REPO_ROOT}"
assert DATA_ROOT.exists(), f"DATA_ROOT 不存在: {DATA_ROOT}"
for case in CASES:
    assert Path(case["case_root"]).expanduser().exists(), f'case_root 不存在: {case["case_root"]}'

print("REPO_ROOT =", REPO_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUT_DIR   =", OUT_DIR)
print("CASES     =", len(CASES))


REPO_ROOT = /root/autodl-tmp/MMRL
DATA_ROOT = /root/autodl-tmp/MMRL/DATASETS
OUT_DIR   = /root/autodl-tmp/MMRL/output_refactor/analysis/dynamic_r_fusion_repo_forward_log_restored
CASES     = 2


In [2]:
import os
import sys
import ast
import math
import json
import random
import re
import importlib
from types import SimpleNamespace
from itertools import product

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from core.config import setup_cfg
from dassl.engine import build_trainer
from core.utils import import_optional_modules

# 与 run.py 对齐：注册 datasets + trainer。
import_optional_modules([
    "datasets.oxford_pets", "datasets.oxford_flowers", "datasets.fgvc_aircraft",
    "datasets.dtd", "datasets.eurosat", "datasets.stanford_cars", "datasets.food101",
    "datasets.sun397", "datasets.caltech101", "datasets.ucf101", "datasets.imagenet",
    "datasets.imagenetv2", "datasets.imagenet_sketch", "datasets.imagenet_a", "datasets.imagenet_r",
])
importlib.import_module("trainers.refactor_runner")

print("torch =", torch.__version__)
print("cuda available =", torch.cuda.is_available())


torch = 2.11.0+cu128
cuda available = True


## 1. 从真实日志恢复配置

`setup_cfg(args)` 会 merge config files 后再应用 `args.opts`。因此真实训练日志中的 `opts` 是模型配置的一部分，不能只靠路径推断。

本节会解析真实日志中的：

```text
***************
** Arguments **
***************
...
opts: [...]
...
************
** Config **
************
```

如果找不到日志或找不到 `Arguments` 区块，会直接报错，避免静默生成错误分支结果。


In [3]:
METHOD_CFG_MAP = {
    "MMRL": "configs/methods/mmrl.yaml",
    "MMRLMix": "configs/methods/mmrl_mix.yaml",
    "MMRLpp": "configs/methods/mmrlpp.yaml",
    "MMRLPP": "configs/methods/mmrlpp.yaml",
    "BayesMMRL": "configs/methods/bayesmmrl.yaml",
    "ClipAdapters": "configs/methods/clip_adapters.yaml",
    "ClipADAPTER": "configs/methods/clip_adapters.yaml",
}

PROTOCOL_CFG_MAP = {
    "B2N": "configs/protocols/b2n.yaml",
    "FS": "configs/protocols/fs.yaml",
    "CD": "configs/protocols/cd.yaml",
}

PROTOCOL_TO_SUBSAMPLE = {
    "B2N": "base",
    "FS": "all",
    "CD": "all",
}

def decode_backbone_from_dir(token: str) -> str:
    # output_refactor 中通常把 ViT-B/16 写成 ViT-B-16。
    if token.startswith("ViT-") and token.count("-") >= 2:
        head, patch = token.rsplit("-", 1)
        return f"{head}/{patch}"
    return token

def infer_case_metadata(case_root: Path):
    case_root = Path(case_root).expanduser().resolve()
    parts = case_root.parts
    if "output_refactor" in parts:
        idx = parts.index("output_refactor")
    elif "output_sweeps" in parts:
        # sweep 路径通常形如:
        # output_sweeps/.../<stage>/<Method>/<Protocol>/<phase>/<dataset>/shots_.../...
        # 向后找第一个已知 method token。
        method_indices = [i for i, p in enumerate(parts) if p in METHOD_CFG_MAP]
        if not method_indices:
            raise ValueError(f"无法从 sweep 路径解析 method: {case_root}")
        idx = method_indices[0] - 1
    else:
        raise ValueError(f"case_root 中找不到 output_refactor/output_sweeps: {case_root}")

    # 优先按 output_refactor 标准路径解析；失败时保留最少 meta，真实 cfg 由日志恢复。
    meta = {"case_root": case_root}
    try:
        if parts[idx] == "output_refactor":
            method = parts[idx + 1]
            protocol = parts[idx + 2]
            phase = parts[idx + 3]
            dataset = parts[idx + 4]
            shots_str = parts[idx + 5]
            backbone_token = parts[idx + 6]
            tag = parts[idx + 7]
            seed_dir = parts[idx + 8]
        else:
            method = parts[idx + 1]
            protocol = parts[idx + 2]
            phase = parts[idx + 3]
            dataset = parts[idx + 4]
            shots_str = parts[idx + 5]
            backbone_token = parts[idx + 6]
            tag = parts[idx + 7]
            seed_dir = parts[idx + 8]

        meta.update({
            "method": method,
            "protocol": protocol,
            "phase": phase,
            "dataset": dataset,
            "shots": int(shots_str.split("_", 1)[1]) if shots_str.startswith("shots_") else None,
            "backbone": decode_backbone_from_dir(backbone_token),
            "tag": tag,
            "seed": int(seed_dir.replace("seed", "")) if seed_dir.startswith("seed") else None,
        })
    except Exception as exc:
        meta["path_parse_warning"] = repr(exc)

    return meta

def find_real_log_file(case_root: Path, explicit_log_file=None):
    if explicit_log_file:
        p = Path(explicit_log_file).expanduser().resolve()
        if p.exists() and p.is_file():
            return p
        raise FileNotFoundError(f"显式指定的 log_file 不存在: {p}")

    case_root = Path(case_root).expanduser().resolve()
    candidates = [
        case_root / "run.log",
        case_root / "log.txt",
    ]

    # log.txt-2026-... 一般是历史复制，按 mtime 逆序尝试。
    candidates += sorted(case_root.glob("log.txt-*"), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)
    candidates += sorted(case_root.glob("*.log"), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)

    seen = set()
    unique_candidates = []
    for p in candidates:
        if str(p) not in seen:
            unique_candidates.append(p)
            seen.add(str(p))

    for p in unique_candidates:
        if p.exists() and p.is_file():
            text_head = p.read_text(encoding="utf-8", errors="ignore")[:20000]
            if "** Arguments **" in text_head and "opts:" in text_head:
                return p

    raise FileNotFoundError(
        f"找不到包含真实 Arguments/opts 的训练日志: {case_root}/run.log 或 log.txt。"
        "本 notebook 不允许只从路径猜配置。"
    )

def _parse_scalar_from_log_value(value: str):
    value = value.strip()
    if value in {"", "None"}:
        return None if value == "None" else ""
    if value in {"True", "False"}:
        return value == "True"
    # opts 必须按 Python literal 解析。
    if value.startswith("[") and value.endswith("]"):
        return ast.literal_eval(value)
    # seed/load_epoch 等单独处理；其他保留字符串，和 argparse 行为一致。
    return value

def parse_args_from_log(log_path: Path):
    text = Path(log_path).read_text(encoding="utf-8", errors="ignore")

    m = re.search(
        r"\*\* Arguments \*\*\s*\n\*+\s*\n(?P<body>.*?)(?:\n\*{4,}\s*\n\*\* Config \*\*|\nCollecting env info|\nLoading trainer:)",
        text,
        flags=re.S,
    )
    if m is None:
        raise ValueError(f"日志中找不到 Arguments 区块: {log_path}")

    arg_text = m.group("body")
    parsed = {}

    # Arguments 区块一般一行一个 "key: value"。
    for raw_line in arg_text.splitlines():
        line = raw_line.rstrip()
        if not line or ": " not in line:
            continue
        key, value = line.split(": ", 1)
        key = key.strip()
        value = value.strip()
        parsed[key] = _parse_scalar_from_log_value(value)

    if "opts" in parsed and isinstance(parsed["opts"], str):
        parsed["opts"] = ast.literal_eval(parsed["opts"])

    # 类型归一化。
    for key in ["eval_only", "no_train"]:
        if key in parsed and isinstance(parsed[key], str):
            parsed[key] = parsed[key] == "True"

    for key in ["seed", "load_epoch"]:
        if key in parsed:
            if parsed[key] in {None, ""}:
                parsed[key] = None
            else:
                parsed[key] = int(parsed[key])

    required = [
        "dataset_config_file",
        "method_config_file",
        "protocol_config_file",
        "runtime_config_file",
        "method",
        "protocol",
        "exec_mode",
        "seed",
        "trainer",
        "opts",
    ]
    missing = [k for k in required if k not in parsed]
    if missing:
        raise ValueError(f"日志 Arguments 缺少字段 {missing}: {log_path}")

    if not isinstance(parsed["opts"], list):
        raise TypeError(f"日志 opts 不是 list: {type(parsed['opts'])}, log={log_path}")

    return parsed

def make_args_from_case(case):
    case_root = Path(case["case_root"]).expanduser().resolve()
    meta = infer_case_metadata(case_root)

    log_path = find_real_log_file(case_root, case.get("log_file", None))
    logged = parse_args_from_log(log_path)

    # 注意：
    # - root 用当前机器的 DATA_ROOT，避免日志中的相对路径 DATASETS 在 notebook 环境下失效。
    # - output_dir/model_dir 用当前 case_root，确保加载当前 checkpoint。
    args = SimpleNamespace(
        root=str(DATA_ROOT),
        output_dir=str(case_root),
        dataset_config_file=str(logged["dataset_config_file"]),
        method_config_file=str(logged["method_config_file"]),
        protocol_config_file=str(logged["protocol_config_file"]),
        runtime_config_file=str(logged["runtime_config_file"]),
        exp_config=str(logged.get("exp_config", "") or ""),
        method=str(logged["method"]),
        protocol=str(logged["protocol"]),
        exec_mode=str(logged["exec_mode"]),
        seed=int(logged["seed"]) if logged.get("seed") is not None else -1,
        trainer=str(logged.get("trainer", "RefactorRunner")),
        eval_only=True,
        model_dir=str(case_root),
        load_epoch=case.get("load_epoch", None),
        no_train=True,
        opts=list(logged["opts"]),
    )

    meta.update({
        "name": case.get("name", case_root.name),
        "log_path": str(log_path),
        "opts_restored_from_log": True,
        "logged_method": logged["method"],
        "logged_protocol": logged["protocol"],
        "logged_exec_mode": logged["exec_mode"],
        "logged_seed": logged["seed"],
        "logged_opts": list(logged["opts"]),
        "logged_output_dir": logged.get("output_dir", ""),
        "logged_root": logged.get("root", ""),
    })

    if REQUIRE_REAL_LOG_CONFIG:
        assert meta["opts_restored_from_log"], "必须从真实日志恢复 opts；不能仅从路径猜配置。"

    return args, meta

def build_trainer_for_case(case):
    args, meta = make_args_from_case(case)
    cfg = setup_cfg(args)
    trainer = build_trainer(cfg)
    trainer.load_model(str(meta["case_root"]), epoch=args.load_epoch)
    trainer.set_model_mode("eval")
    return trainer, meta, args

def get_cfg_alpha(trainer):
    cfg = trainer.cfg
    if str(cfg.METHOD.NAME) == "BayesMMRL":
        return float(cfg.BAYES_MMRL.ALPHA)
    if hasattr(cfg, "MMRL") and hasattr(cfg.MMRL, "ALPHA"):
        return float(cfg.MMRL.ALPHA)
    if hasattr(trainer.method, "method_cfg") and hasattr(trainer.method.method_cfg, "ALPHA"):
        return float(trainer.method.method_cfg.ALPHA)
    return float("nan")


In [4]:
def audit_trainer_config(trainer, meta, args, case_name):
    cfg = trainer.cfg
    rows = []

    def add(key, value, expected=None):
        rows.append({
            "case_name": case_name,
            "key": key,
            "value": str(value),
            "expected_or_inferred": "" if expected is None else str(expected),
            "match": "" if expected is None else bool(str(value) == str(expected)),
        })

    add("log_path", meta.get("log_path", ""))
    add("opts_restored_from_log", meta.get("opts_restored_from_log", False), True)
    add("logged_output_dir", meta.get("logged_output_dir", ""))
    add("logged_root", meta.get("logged_root", ""))
    add("cfg.METHOD.NAME", cfg.METHOD.NAME, meta.get("logged_method", meta.get("method", "")))
    add("cfg.METHOD.EXEC_MODE", cfg.METHOD.EXEC_MODE, meta.get("logged_exec_mode", args.exec_mode))
    add("cfg.PROTOCOL.NAME", cfg.PROTOCOL.NAME, meta.get("logged_protocol", meta.get("protocol", "")))
    add("cfg.PROTOCOL.PHASE", getattr(cfg.PROTOCOL, "PHASE", None), meta.get("phase", ""))
    add("cfg.DATASET.NAME", cfg.DATASET.NAME, meta.get("dataset", ""))
    add("cfg.DATASET.NUM_SHOTS", cfg.DATASET.NUM_SHOTS, meta.get("shots", ""))
    add("cfg.DATASET.SUBSAMPLE_CLASSES", cfg.DATASET.SUBSAMPLE_CLASSES)
    add("cfg.MODEL.BACKBONE.NAME", cfg.MODEL.BACKBONE.NAME, meta.get("backbone", ""))
    add("cfg.SEED", cfg.SEED, meta.get("logged_seed", meta.get("seed", "")))
    add("cfg.OUTPUT_DIR", cfg.OUTPUT_DIR, meta["case_root"])

    if str(cfg.METHOD.NAME) == "BayesMMRL":
        keys = [
            "ALPHA", "REG_WEIGHT", "N_REP_TOKENS", "REP_LAYERS", "REP_DIM",
            "BAYES_TARGET", "N_MC_TRAIN", "N_MC_TEST",
            "EVAL_MODE", "EVAL_USE_POSTERIOR_MEAN", "EVAL_AGGREGATION",
            "USE_MEAN_MAIN_MC_REP", "KL_WARMUP_EPOCHS",
            "REP_SIGMA_MODE", "REP_PRIOR_MODE", "REP_PRIOR_STD", "REP_KL_WEIGHT",
            "PROJ_REP_SIGMA_MODE", "PROJ_REP_PRIOR_MODE", "PROJ_REP_PRIOR_STD", "PROJ_REP_KL_WEIGHT",
            "MAIN_CONSISTENCY_WEIGHT", "MAIN_CONSISTENCY_MODE",
        ]
        for k in keys:
            if hasattr(cfg.BAYES_MMRL, k):
                add(f"cfg.BAYES_MMRL.{k}", getattr(cfg.BAYES_MMRL, k))
    elif hasattr(cfg, "MMRL"):
        for k in ["ALPHA", "REG_WEIGHT", "N_REP_TOKENS", "REP_LAYERS", "REP_DIM"]:
            if hasattr(cfg.MMRL, k):
                add(f"cfg.MMRL.{k}", getattr(cfg.MMRL, k))

    add("args.dataset_config_file", args.dataset_config_file)
    add("args.method_config_file", args.method_config_file)
    add("args.protocol_config_file", args.protocol_config_file)
    add("args.runtime_config_file", args.runtime_config_file)
    add("args.exp_config", args.exp_config)
    add("args.opts", args.opts)
    add("logged_opts", meta.get("logged_opts", []))

    return pd.DataFrame(rows)


## 2. Repo-forward 分支收集

这里仍然用项目标准路径：

- `trainer.method.forward_eval(batch, eval_ctx)`
- `outputs.logits` → C/main branch
- `outputs.aux_logits["rep"]` → R/representation branch
- `outputs.aux_logits["fusion"]` → repo fusion branch
- `trainer.method.select_eval_logits(outputs, eval_ctx)` → 和 `trainer.test()` 一致的 eval 路由


In [5]:
def set_eval_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def collect_repo_forward_logits(trainer, split="test", max_batches=None, seed=2026):
    set_eval_seed(seed)
    trainer.set_model_mode("eval")

    if split == "val" and getattr(trainer, "val_loader", None) is not None:
        data_loader = trainer.val_loader
        actual_split = "val"
    else:
        data_loader = trainer.test_loader
        actual_split = "test"

    eval_ctx = trainer.executor.build_eval_context(trainer, actual_split)

    z_c_all, z_r_all, z_aux_fusion_all, z_repo_eval_all, y_all, index_all = [], [], [], [], [], []
    n_seen = 0

    for batch_idx, batch in enumerate(data_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        image = batch["img"].to(trainer.device)
        label = batch["label"].to(trainer.device).long()

        outputs = trainer.method.forward_eval(
            {"img": image, "label": label},
            eval_ctx,
        )

        if outputs.aux_logits is None or "rep" not in outputs.aux_logits or "fusion" not in outputs.aux_logits:
            raise RuntimeError(
                "outputs.aux_logits 必须包含 'rep' 和 'fusion'。"
                f" 当前 aux_logits keys={None if outputs.aux_logits is None else list(outputs.aux_logits.keys())}"
            )

        z_c = outputs.logits
        z_r = outputs.aux_logits["rep"]
        z_aux_fusion = outputs.aux_logits["fusion"]
        z_repo_eval = trainer.method.select_eval_logits(outputs, eval_ctx)

        if not (z_c.shape == z_r.shape == z_aux_fusion.shape == z_repo_eval.shape):
            raise RuntimeError(
                f"logits shape 不一致: z_c={tuple(z_c.shape)}, z_r={tuple(z_r.shape)}, "
                f"z_aux_fusion={tuple(z_aux_fusion.shape)}, z_repo_eval={tuple(z_repo_eval.shape)}"
            )

        bsz = int(label.shape[0])
        z_c_all.append(z_c.detach().float().cpu())
        z_r_all.append(z_r.detach().float().cpu())
        z_aux_fusion_all.append(z_aux_fusion.detach().float().cpu())
        z_repo_eval_all.append(z_repo_eval.detach().float().cpu())
        y_all.append(label.detach().cpu())
        index_all.append(torch.arange(n_seen, n_seen + bsz, dtype=torch.long))
        n_seen += bsz

    if not y_all:
        raise RuntimeError(f"No batches collected for split={split}")

    return {
        "actual_split": actual_split,
        "z_c": torch.cat(z_c_all, dim=0).numpy(),
        "z_r": torch.cat(z_r_all, dim=0).numpy(),
        "z_aux_fusion": torch.cat(z_aux_fusion_all, dim=0).numpy(),
        "z_repo_eval": torch.cat(z_repo_eval_all, dim=0).numpy(),
        "y": torch.cat(y_all, dim=0).numpy(),
        "index": torch.cat(index_all, dim=0).numpy(),
    }

def cache_path_for(case_name, split):
    safe = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in case_name)
    return OUT_DIR / f"{safe}_{split}_repo_forward_logits_log_restored.npz"

def load_or_collect_case_split(trainer, case_name, split, max_batches=None, seed=2026):
    path = cache_path_for(case_name, split)
    if USE_NPZ_CACHE_IF_EXISTS and path.exists():
        data = dict(np.load(path, allow_pickle=True))
        data["actual_split"] = str(data["actual_split"].item() if hasattr(data["actual_split"], "item") else data["actual_split"])
        return data

    data = collect_repo_forward_logits(
        trainer=trainer,
        split=split,
        max_batches=max_batches,
        seed=seed,
    )
    if SAVE_NPZ_CACHE:
        np.savez_compressed(path, **data)
    return data


## 3. 指标与动态融合函数


In [6]:
EPS = 1e-12

def softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def log_softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    m = np.max(z, axis=1, keepdims=True)
    logsum = m + np.log(np.sum(np.exp(z - m), axis=1, keepdims=True))
    return z - logsum

def top1_acc_from_logits(z, y):
    return float(np.mean(np.argmax(z, axis=1) == y))

def n_correct_from_logits(z, y):
    return int(np.sum(np.argmax(z, axis=1) == y))

def nll_from_logits(z, y):
    logp = log_softmax_np(z)
    return float(-np.mean(logp[np.arange(len(y)), y]))

def brier_from_logits(z, y):
    p = softmax_np(z)
    onehot = np.zeros_like(p)
    onehot[np.arange(len(y)), y] = 1.0
    return float(np.mean(np.sum((p - onehot) ** 2, axis=1)))

def ece_from_logits(z, y, n_bins=15):
    p = softmax_np(z)
    conf = np.max(p, axis=1)
    pred = np.argmax(p, axis=1)
    correct = (pred == y).astype(np.float64)

    ece = 0.0
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == 0:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf > lo) & (conf <= hi)
        if np.any(mask):
            ece += np.mean(mask) * abs(np.mean(correct[mask]) - np.mean(conf[mask]))
    return float(ece)

def entropy_np(p):
    p = np.asarray(p, dtype=np.float64)
    return -np.sum(np.clip(p, EPS, 1.0) * np.log(np.clip(p, EPS, 1.0)), axis=1)

def margin_np(p):
    vals = np.sort(p, axis=1)
    if vals.shape[1] == 1:
        return vals[:, -1]
    return vals[:, -1] - vals[:, -2]

def js_divergence_np(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    m = 0.5 * (p + q)
    return 0.5 * np.sum(np.clip(p, EPS, 1.0) * (np.log(np.clip(p, EPS, 1.0)) - np.log(np.clip(m, EPS, 1.0))), axis=1) + \
           0.5 * np.sum(np.clip(q, EPS, 1.0) * (np.log(np.clip(q, EPS, 1.0)) - np.log(np.clip(m, EPS, 1.0))), axis=1)

def branch_metrics_row(case_name, split, branch_name, z, y):
    return {
        "case_name": case_name,
        "split": split,
        "branch": branch_name,
        "n": int(len(y)),
        "n_correct": n_correct_from_logits(z, y),
        "accuracy": top1_acc_from_logits(z, y),
        "accuracy_pct": 100.0 * top1_acc_from_logits(z, y),
        "nll": nll_from_logits(z, y),
        "brier": brier_from_logits(z, y),
        "ece": ece_from_logits(z, y),
    }

def compute_dynamic_components(z_c, z_r):
    p_c = softmax_np(z_c)
    p_r = softmax_np(z_r)
    c = z_c.shape[1]

    ent_c = entropy_np(p_c) / max(math.log(c), EPS)
    margin_c = margin_np(p_c)
    margin_r = margin_np(p_r)
    js = js_divergence_np(p_c, p_r) / max(math.log(2.0), EPS)

    # 希望 R 权重随 C 不确定性增加而增加、随 R 相对 margin 增加而增加、随分歧适度增加。
    margin_gain = margin_r - margin_c

    return {
        "p_c": p_c,
        "p_r": p_r,
        "u_c": ent_c,
        "margin_c": margin_c,
        "margin_r": margin_r,
        "margin_gain": margin_gain,
        "js": js,
    }

def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60.0, 60.0)))

def dynamic_weights_no_beta(z_c, z_r):
    comp = compute_dynamic_components(z_c, z_r)
    score = 0.5 * comp["u_c"] + 0.25 * comp["margin_gain"] + 0.25 * comp["js"]
    # 规则式权重约束在 [0, 1]。
    omega = np.clip(score, 0.0, 1.0)
    return omega, comp

def dynamic_weights_beta(z_c, z_r, beta_u=1.0, beta_m=1.0, beta_d=1.0, bias=0.0):
    comp = compute_dynamic_components(z_c, z_r)
    score = bias + beta_u * comp["u_c"] + beta_m * comp["margin_gain"] + beta_d * comp["js"]
    omega = sigmoid_np(score)
    return omega, comp

def fuse_dynamic_logits(z_c, z_r, omega_r):
    omega_r = np.asarray(omega_r, dtype=np.float64).reshape(-1, 1)
    return (1.0 - omega_r) * z_c + omega_r * z_r

def summarize_logits_matrix(prefix, z):
    return {
        f"{prefix}_shape": str(tuple(z.shape)),
        f"{prefix}_mean": float(np.mean(z)),
        f"{prefix}_std": float(np.std(z)),
        f"{prefix}_min": float(np.min(z)),
        f"{prefix}_max": float(np.max(z)),
    }


In [7]:
def evaluate_dynamic_variants(case_name, split, data, beta=None):
    y = data["y"]
    z_c = data["z_c"]
    z_r = data["z_r"]

    rows = []
    omega0, comp0 = dynamic_weights_no_beta(z_c, z_r)
    z_dyn0 = fuse_dynamic_logits(z_c, z_r, omega0)
    row0 = branch_metrics_row(case_name, split, "dynamic_no_beta", z_dyn0, y)
    row0.update({
        "omega_mean": float(np.mean(omega0)),
        "omega_std": float(np.std(omega0)),
        "omega_min": float(np.min(omega0)),
        "omega_max": float(np.max(omega0)),
        "beta_u": "",
        "beta_m": "",
        "beta_d": "",
        "bias": "",
    })
    rows.append(row0)

    if beta is not None:
        omega_b, comp_b = dynamic_weights_beta(z_c, z_r, **beta)
        z_dyn_b = fuse_dynamic_logits(z_c, z_r, omega_b)
        rowb = branch_metrics_row(case_name, split, "dynamic_beta_selected_on_val", z_dyn_b, y)
        rowb.update({
            "omega_mean": float(np.mean(omega_b)),
            "omega_std": float(np.std(omega_b)),
            "omega_min": float(np.min(omega_b)),
            "omega_max": float(np.max(omega_b)),
            **beta,
        })
        rows.append(rowb)

    return rows

def grid_search_beta_on_split(case_name, data_val, grid_values):
    z_c = data_val["z_c"]
    z_r = data_val["z_r"]
    y = data_val["y"]

    rows = []
    best = None
    # 加一个 bias 网格，允许整体降低/提高 R 权重。
    bias_grid = [-4.0, -2.0, 0.0, 2.0, 4.0]

    for beta_u, beta_m, beta_d, bias in product(grid_values, grid_values, grid_values, bias_grid):
        beta = {
            "beta_u": float(beta_u),
            "beta_m": float(beta_m),
            "beta_d": float(beta_d),
            "bias": float(bias),
        }
        omega, _ = dynamic_weights_beta(z_c, z_r, **beta)
        z_dyn = fuse_dynamic_logits(z_c, z_r, omega)
        acc = top1_acc_from_logits(z_dyn, y)
        nll = nll_from_logits(z_dyn, y)
        row = {
            "case_name": case_name,
            "split": data_val["actual_split"],
            **beta,
            "accuracy": acc,
            "accuracy_pct": 100.0 * acc,
            "n_correct": n_correct_from_logits(z_dyn, y),
            "nll": nll,
            "omega_mean": float(np.mean(omega)),
            "omega_std": float(np.std(omega)),
        }
        rows.append(row)
        if best is None or (acc, -nll) > (best["accuracy"], -best["nll"]):
            best = row

    return best, pd.DataFrame(rows).sort_values(["accuracy", "nll"], ascending=[False, True])


## 4. 主流程：构建 trainer、收集 logits、生成报告


In [ ]:
all_accuracy_rows = []
all_sanity_rows = []
all_config_audit_rows = []
all_beta_top_rows = []
all_full_diag_rows = []
all_transition_rows = []
all_changed_rows = []
case_results = {}

for case in CASES:
    case_name = case["name"]
    print("\n" + "=" * 80)
    print("CASE:", case_name)
    print("=" * 80)

    trainer, meta, args = build_trainer_for_case(case)
    print("restored log:", meta.get("log_path"))
    print("restored opts count:", len(args.opts))

    audit_df = audit_trainer_config(trainer, meta, args, case_name)
    all_config_audit_rows.append(audit_df)

    test_data = load_or_collect_case_split(
        trainer,
        case_name,
        split=TEST_SPLIT,
        max_batches=MAX_BATCHES,
        seed=EVAL_SEED,
    )
    case_results[(case_name, "test")] = test_data

    # 尽量收集 val 以选择 beta；如果 val 不存在，会回退 test，这会在 actual_split 中显示出来。
    val_data = load_or_collect_case_split(
        trainer,
        case_name,
        split=BETA_SELECT_SPLIT,
        max_batches=MAX_BATCHES,
        seed=EVAL_SEED,
    )
    case_results[(case_name, "val_or_test")] = val_data

    for split_name, data in [("test", test_data), ("beta_select", val_data)]:
        y = data["y"]
        actual_split = data["actual_split"]

        all_accuracy_rows.extend([
            branch_metrics_row(case_name, actual_split, "C_only_outputs_logits", data["z_c"], y),
            branch_metrics_row(case_name, actual_split, "R_only_aux_logits_rep", data["z_r"], y),
            branch_metrics_row(case_name, actual_split, "repo_aux_logits_fusion", data["z_aux_fusion"], y),
            branch_metrics_row(case_name, actual_split, "repo_eval_select_eval_logits", data["z_repo_eval"], y),
        ])

        alpha = get_cfg_alpha(trainer)
        if np.isfinite(alpha):
            z_formula = alpha * data["z_c"] + (1.0 - alpha) * data["z_r"]
            diff_formula = np.max(np.abs(z_formula - data["z_aux_fusion"]))
        else:
            diff_formula = float("nan")

        diff_repo_fusion_eval = np.max(np.abs(data["z_repo_eval"] - data["z_aux_fusion"]))

        all_sanity_rows.append({
            "case_name": case_name,
            "requested_split_alias": split_name,
            "actual_split": actual_split,
            "n": int(len(y)),
            "alpha": alpha,
            "max_abs_formula_alpha_vs_aux_fusion": float(diff_formula),
            "formula_alpha_matches_aux_fusion_atol_1e-4": bool(np.isfinite(diff_formula) and diff_formula < 1e-4),
            "max_abs_repo_eval_vs_aux_fusion": float(diff_repo_fusion_eval),
            "repo_eval_equals_aux_fusion_atol_1e-4": bool(diff_repo_fusion_eval < 1e-4),
            **summarize_logits_matrix("z_c", data["z_c"]),
            **summarize_logits_matrix("z_r", data["z_r"]),
            **summarize_logits_matrix("z_aux_fusion", data["z_aux_fusion"]),
            **summarize_logits_matrix("z_repo_eval", data["z_repo_eval"]),
        })

        pred_c = np.argmax(data["z_c"], axis=1)
        pred_r = np.argmax(data["z_r"], axis=1)
        pred_f = np.argmax(data["z_aux_fusion"], axis=1)
        pred_eval = np.argmax(data["z_repo_eval"], axis=1)

        all_transition_rows.append({
            "case_name": case_name,
            "actual_split": actual_split,
            "n": int(len(y)),
            "C_correct_R_correct": int(np.sum((pred_c == y) & (pred_r == y))),
            "C_correct_R_wrong": int(np.sum((pred_c == y) & (pred_r != y))),
            "C_wrong_R_correct": int(np.sum((pred_c != y) & (pred_r == y))),
            "C_wrong_R_wrong": int(np.sum((pred_c != y) & (pred_r != y))),
            "C_pred_eq_R_pred": int(np.sum(pred_c == pred_r)),
            "C_pred_neq_R_pred": int(np.sum(pred_c != pred_r)),
            "fusion_changes_C_prediction": int(np.sum(pred_f != pred_c)),
            "repo_eval_changes_C_prediction": int(np.sum(pred_eval != pred_c)),
        })

        changed_idx = np.where(pred_f != pred_c)[0]
        if len(changed_idx) > 0:
            p_c = softmax_np(data["z_c"])
            p_r = softmax_np(data["z_r"])
            p_f = softmax_np(data["z_aux_fusion"])
            for i in changed_idx[:5000]:
                all_changed_rows.append({
                    "case_name": case_name,
                    "actual_split": actual_split,
                    "index": int(data["index"][i]),
                    "label": int(y[i]),
                    "pred_c": int(pred_c[i]),
                    "pred_r": int(pred_r[i]),
                    "pred_fusion": int(pred_f[i]),
                    "c_correct": bool(pred_c[i] == y[i]),
                    "r_correct": bool(pred_r[i] == y[i]),
                    "fusion_correct": bool(pred_f[i] == y[i]),
                    "conf_c": float(np.max(p_c[i])),
                    "conf_r": float(np.max(p_r[i])),
                    "conf_fusion": float(np.max(p_f[i])),
                })

        comp = compute_dynamic_components(data["z_c"], data["z_r"])
        diag_n = len(y)
        for i in range(diag_n):
            all_full_diag_rows.append({
                "case_name": case_name,
                "actual_split": actual_split,
                "index": int(data["index"][i]),
                "label": int(y[i]),
                "pred_c": int(pred_c[i]),
                "pred_r": int(pred_r[i]),
                "pred_fusion": int(pred_f[i]),
                "pred_repo_eval": int(pred_eval[i]),
                "c_correct": bool(pred_c[i] == y[i]),
                "r_correct": bool(pred_r[i] == y[i]),
                "fusion_correct": bool(pred_f[i] == y[i]),
                "repo_eval_correct": bool(pred_eval[i] == y[i]),
                "u_c": float(comp["u_c"][i]),
                "margin_c": float(comp["margin_c"][i]),
                "margin_r": float(comp["margin_r"][i]),
                "margin_gain": float(comp["margin_gain"][i]),
                "js_c_r": float(comp["js"][i]),
            })

    # beta 只用 val/beta_select split 选择，然后评估到 test。
    best_beta, beta_grid_df = grid_search_beta_on_split(case_name, val_data, BETA_GRID_VALUES)
    all_beta_top_rows.append(beta_grid_df.head(50))
    all_accuracy_rows.extend(evaluate_dynamic_variants(case_name, test_data["actual_split"], test_data, beta=best_beta))

    print("test n:", len(test_data["y"]), "actual_split:", test_data["actual_split"])
    print("C acc:", top1_acc_from_logits(test_data["z_c"], test_data["y"]))
    print("R acc:", top1_acc_from_logits(test_data["z_r"], test_data["y"]))
    print("Fusion acc:", top1_acc_from_logits(test_data["z_aux_fusion"], test_data["y"]))
    print("Repo eval acc:", top1_acc_from_logits(test_data["z_repo_eval"], test_data["y"]))
    print("best beta:", best_beta)

config_audit_df = pd.concat(all_config_audit_rows, ignore_index=True) if all_config_audit_rows else pd.DataFrame()
accuracy_df = pd.DataFrame(all_accuracy_rows)
sanity_df = pd.DataFrame(all_sanity_rows)
transitions_df = pd.DataFrame(all_transition_rows)
changed_df = pd.DataFrame(all_changed_rows)
full_diag_df = pd.DataFrame(all_full_diag_rows)
beta_top_df = pd.concat(all_beta_top_rows, ignore_index=True) if all_beta_top_rows else pd.DataFrame()

print("\nAccuracy:")
display(accuracy_df)
print("\nSanity checks:")
display(sanity_df)



CASE: MMRL_16shot_seed1
Loading trainer: RefactorRunner
Loading dataset: Caltech101
Reading split from /root/autodl-tmp/MMRL/DATASETS/caltech-101/split_zhou_Caltech101.json
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/caltech-101/split_fewshot/shot_16-seed_1.pkl
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.5, 1))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
---------  ----------
Dataset    Caltech101
# classes  100
# train_x  1,600
# val      400
# test     2,465
---------  ----------
[MMRL] trainable params: {'representation_learner.compound_rep_tokens', 'representation_learner.compound_rep_tokens_r2vproj.5.bias', 're

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:72: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None


TypeError: dynamic_weights_beta() got an unexpected keyword argument 'case_name'

: 

## 5. 结论与 Excel 输出


In [ ]:
def build_conclusions_df(accuracy_df, sanity_df, config_audit_df):
    rows = []

    rows.append({
        "item": "root_cause",
        "conclusion": "本版修复点是从真实训练日志恢复完整 args/opts；不再只从路径猜配置。",
        "detail": "如果 z_c/z_r 准确率异常低，优先检查 ConfigAudit 中 log_path、opts_restored_from_log、args.opts 与真实 run.log 是否一致。",
    })

    if not sanity_df.empty:
        for _, r in sanity_df.iterrows():
            rows.append({
                "item": "sanity",
                "conclusion": f"{r['case_name']} / {r['actual_split']} / n={r['n']}",
                "detail": (
                    f"alpha={r['alpha']}; "
                    f"max_abs_formula_alpha_vs_aux_fusion={r['max_abs_formula_alpha_vs_aux_fusion']}; "
                    f"max_abs_repo_eval_vs_aux_fusion={r['max_abs_repo_eval_vs_aux_fusion']}"
                ),
            })

    if not accuracy_df.empty:
        test_rows = accuracy_df[accuracy_df["split"].eq("test")].copy()
        for _, r in test_rows.iterrows():
            rows.append({
                "item": "accuracy",
                "conclusion": f"{r['case_name']} / {r['branch']}: {r['accuracy_pct']:.3f}%",
                "detail": f"n_correct={r['n_correct']} / n={r['n']}, nll={r['nll']:.6f}, brier={r['brier']:.6f}, ece={r['ece']:.6f}",
            })

    if not config_audit_df.empty:
        bad = config_audit_df[config_audit_df["key"].eq("opts_restored_from_log")]
        for _, r in bad.iterrows():
            rows.append({
                "item": "config_restore",
                "conclusion": f"{r['case_name']}: opts_restored_from_log={r['value']}",
                "detail": f"expected={r['expected_or_inferred']}, match={r['match']}",
            })

    return pd.DataFrame(rows)

conclusions_df = build_conclusions_df(accuracy_df, sanity_df, config_audit_df)

readme_df = pd.DataFrame([
    {
        "sheet": "README",
        "description": "说明各个 sheet 的含义。",
    },
    {
        "sheet": "Conclusions",
        "description": "自动生成的结论摘要，重点检查真实日志配置恢复和 test 分支准确率。",
    },
    {
        "sheet": "ConfigAudit",
        "description": "每个 case 从真实日志恢复的 cfg/args/opts 审计。重点看 log_path、opts_restored_from_log、args.opts。",
    },
    {
        "sheet": "Accuracy",
        "description": "C 分支、R 分支、repo fusion、repo eval、动态融合的 accuracy/NLL/Brier/ECE。",
    },
    {
        "sheet": "SanityChecks",
        "description": "shape/logit 范围、alpha 公式与 aux_fusion 的差异、repo_eval 与 aux_fusion 差异。",
    },
    {
        "sheet": "Transitions",
        "description": "C/R 分支正确性组合、fusion 是否改变 C 分支预测。",
    },
    {
        "sheet": "BetaTop",
        "description": "在 BETA_SELECT_SPLIT 上搜索出的动态融合 beta top candidates。",
    },
    {
        "sheet": "ChangedSamples",
        "description": "fusion 相对 C 分支改变预测的样本，最多保存前 5000 个。",
    },
    {
        "sheet": "FullDiagnostics",
        "description": "逐样本预测、正确性、u/margin/JS 诊断量。",
    },
])

report_path = OUT_DIR / "dynamic_r_fusion_repo_forward_log_restored_report.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    readme_df.to_excel(writer, sheet_name="README", index=False)
    conclusions_df.to_excel(writer, sheet_name="Conclusions", index=False)
    config_audit_df.to_excel(writer, sheet_name="ConfigAudit", index=False)
    accuracy_df.to_excel(writer, sheet_name="Accuracy", index=False)
    sanity_df.to_excel(writer, sheet_name="SanityChecks", index=False)
    transitions_df.to_excel(writer, sheet_name="Transitions", index=False)
    beta_top_df.to_excel(writer, sheet_name="BetaTop", index=False)
    changed_df.to_excel(writer, sheet_name="ChangedSamples", index=False)
    full_diag_df.to_excel(writer, sheet_name="FullDiagnostics", index=False)

print("Saved report:", report_path)
display(conclusions_df)


## 6. 快速人工校验

如果你期望 Caltech101 test set 为 `n=2465`，可以运行：

```python
assert sanity_df.query("actual_split == 'test'")["n"].eq(2465).all()
```

如果 R 分支不是你预期的 95% 左右，先看：

1. `ConfigAudit.log_path` 是否指向真实训练日志；
2. `ConfigAudit.args.opts` 是否包含真实训练命令里的所有覆盖项；
3. `Accuracy` 的 `split` 是否为 `test`、`n` 是否为 2465；
4. `SanityChecks.max_abs_formula_alpha_vs_aux_fusion` 是否符合该方法当前 eval aggregation 的预期。
